# Deteccao de Malware com GANs — Aula 7: TON_IoT (dados reais) + 4 classificadores

**Expansao multi-dataset do estudo "Reducing False Negatives in IoT Malware Detection"**

**SÉRIE DE AULAS 6–8 — Expansão multi-dataset (datasets IoT/IIoT reais)**

As aulas 1–5 (IoT-23) testaram 6 estratégias de balanceamento com um **MLP**. Para fortalecer o artigo (revisores exigem múltiplos datasets IoT e múltiplos classificadores), as aulas 6 e 7 repetem **exatamente o mesmo protocolo** em dois datasets públicos **IoT/IIoT reais** — **Edge-IIoTset** e **TON_IoT** — e adicionam **XGBoost, RandomForest e LSTM** ao lado do MLP. A aula 8 consolida tudo.

**DADOS MANTIDOS (protocolo idêntico ao IoT-23):** amostra estratificada de **40.000 registros** com **82% benigno / 18% ataque**, split 70/30 estratificado, `StandardScaler` + `SelectPercentile(60%)`, 6 cenários de balanceamento (Original, SMOTETomek, GAN+MLP, WGAN-GP, cWGAN-GP, CTGAN) e o **mesmo teste** para todos. Qualquer diferença de métrica vem do balanceamento, não dos dados.

**NOTA DE HONESTIDADE CIENTÍFICA:** usamos o **dataset real** baixado de fonte pública (mirror HuggingFace do dataset oficial UNSW). A amostragem estratificada para 82/18 é declarada no artigo, tornando os três datasets diretamente comparáveis.

## O dataset desta aula: TON_IoT (rede)

- **Origem:** Alsaedi, Moustafa, Tari, Mahmood e Anwar, *IEEE Access*, 2020. Coletado no Cyber Range / IoT Lab da UNSW Canberra (Australia), representando uma rede de escala media com camadas IoT, IIoT, Cloud e Edge/Fog.
- **Arquitetura:** 44 colunas de fluxo no estilo Zeek/Argus (`duration`, `src_bytes`, `dst_bytes`, `conn_state`, `src_pkts`, `dst_pkts`, `missed_bytes`, DNS/TLS/HTTP/`weird`), + `label` binario (0 benigno / 1 ataque) + `type` (categoria: backdoor, ddos, dos, injection, password, ransomware, scanning, xss, mitm).
- **Acesso:** `train_test_network.csv` via mirror publico no HuggingFace (com fallback); fonte oficial em research.unsw.edu.au/projects/toniot-datasets.
- **Uso neste estudo:** amostra estratificada de 40.000 registros com **82% benigno / 18% ataque**, split 70/30, `SelectPercentile(60%)`.

> **Citacao:** A. Alsaedi, N. Moustafa, Z. Tari, A. Mahmood and A. Anwar, "TON_IoT Telemetry Dataset: A New Generation Dataset of IoT and IIoT for Data-Driven Intrusion Detection Systems," *IEEE Access*, vol. 8, pp. 165130-165150, 2020, doi: 10.1109/ACCESS.2020.3022862.


In [ ]:
# ========================================
# CELULA 1: Instala dependencias + seeds
# ========================================
!pip install -q imbalanced-learn xgboost tensorflow scikit-learn matplotlib seaborn ctgan

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

np.random.seed(42)
tf.random.set_seed(42)
print("Dependencias carregadas!")


In [ ]:
# ========================================
# CELULA 2: Helpers de download (com fallback)
# ========================================
import os, time
from urllib.request import urlopen, Request

def baixar(url, destino, tentativas=3):
    """Baixa com header de navegador e progresso simples; tenta N vezes."""
    req = Request(url, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept": "*/*"})
    for t in range(1, tentativas + 1):
        try:
            with urlopen(req, timeout=120) as r:
                total = int(r.headers.get("Content-Length") or 0)
                lido = 0
                with open(destino, "wb") as f:
                    while True:
                        bloco = r.read(1 << 20)
                        if not bloco:
                            break
                        f.write(bloco)
                        lido += len(bloco)
                        if total:
                            pct = 100 * lido / total
                            print(f"\r  {lido/1e6:.1f}/{total/1e6:.0f} MB ({pct:.0f}%)", end="")
            print("")
            print(f"  OK: {destino} ({os.path.getsize(destino)/1e6:.1f} MB)")
            return True
        except Exception as e:
            print(f"  Tentativa {t} falhou: {e}")
            time.sleep(3)
    return False


In [ ]:
# ========================================
# CELULA 3: Download do TON_IoT (rede) - train_test_network.csv
# ========================================
# Fonte oficial: UNSW Canberra / Cyber Range Lab (research.unsw.edu.au/projects/toniot-datasets).
# Mirror publico no HuggingFace (com fallback). Se falhar, baixe
# train_test_network.csv manualmente e faca upload para /content/.
URLS = [
    "https://huggingface.co/datasets/codymlewis/TON_IoT_network/resolve/main/train_test_network.csv",
]
ARQUIVO = "/content/train_test_network.csv"

if not os.path.exists(ARQUIVO) or os.path.getsize(ARQUIVO) < 1e6:
    ok = False
    for url in URLS:
        print("Baixando:", url)
        if baixar(url, ARQUIVO):
            ok = True
            break
    if not ok:
        print("\nDownload falhou. Baixe train_test_network.csv manualmente e faca upload para /content/.")
        raise SystemExit("Download falhou.")
else:
    print("Arquivo ja presente:", ARQUIVO)

df = pd.read_csv(ARQUIVO, low_memory=False)
print("Dimensoes:", df.shape)
print("Colunas:", list(df.columns))
print("\nExemplo das primeiras linhas:")
print(df.head(3).to_string())


In [ ]:
# ========================================
# Funcoes compartilhadas de limpeza / amostragem
# ========================================
def limpar_tabular(df, col_label, mapear):
    """Deixa apenas colunas numericas + label binario."""
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    col = col_label.lower()
    antes = len(df)
    if df[col].dtype == object:
        # normaliza hifens unicode (en-dash/em-dash) usados no CICIDS2017
        df[col] = df[col].str.replace("\u2013", "-", regex=True).str.replace("\u2014", "-", regex=True)
        # remove linhas de cabecalho repetido (label igual ao nome da coluna)
        df = df[df[col].astype(str).str.strip().str.lower() != col]
    # descarta linhas sem label e valores de lixo que viram 'nan'
    df = df[df[col].notna()]
    df[col] = df[col].astype(str).str.strip().str.lower()
    df = df[~df[col].isin(["nan", "none", "na", "n/a", "-", "label"])]
    # mapeia e REMOVE (com aviso) qualquer label nao mapeado - nao aborta mais
    df['label'] = df[col].map(mapear)
    nao_mapeadas = df[df['label'].isna()]
    if len(nao_mapeadas):
        print("  AVISO limpar_tabular: %d linhas removidas por label nao mapeado/NaN." % len(nao_mapeadas))
        unicos = nao_mapeadas[col].unique()[:10]
        print("  Labels removidos:", unicos)
        df = df[df['label'].notna()]
    print("  Linhas: %d -> %d após limpeza (%d removidas)." % (antes, len(df), antes - len(df)))
    if col != 'label':
        df = df.drop(columns=[col])
    if 'id' in df.columns:
        df = df.drop(columns=['id'])
    X = df.drop(columns=['label']).apply(pd.to_numeric, errors='coerce')
    X = X.dropna(axis=1, how='all')
    X = X.loc[:, X.notna().mean() > 0.7]
    X = X.loc[:, X.nunique() > 1]
    X = X.fillna(0)
    X = X.replace([np.inf, -np.inf], 0)
    # compacta para float32: corta a RAM pela metade. Valores que estouram a
    # faixa do float32 viram inf no cast -> zerados na sequencia (mesma
    # politica de tratamento de inf adotada acima).
    import warnings as _w
    with _w.catch_warnings():
        _w.simplefilter('ignore', RuntimeWarning)
        X = X.astype('float32')
    X = X.replace([np.inf, -np.inf], 0)
    y = df['label'].astype(int).values
    del df
    return X, y

def amostra_estratificada(X, y, n_total, frac_benigno, seed=42):
    """Amostra estratificada para espelhar o protocolo do IoT-23 (82% benigno / 18% ataque)."""
    n_ben = int(n_total * frac_benigno)
    n_att = n_total - n_ben
    n_ben_disp = int(sum(y == 0)); n_att_disp = int(sum(y == 1))
    if n_ben > n_ben_disp or n_att > n_att_disp:
        raise ValueError(
            f"Populacao insuficiente para 82/18: disponivel benigno={n_ben_disp}, "
            f"ataque={n_att_disp}; necessario benigno={n_ben}, ataque={n_att}. "
            f"Verifique se TODOS os arquivos do dataset foram baixados.")
    idx_ben = np.random.RandomState(seed).choice(np.where(y == 0)[0], n_ben, replace=False)
    idx_att = np.random.RandomState(seed).choice(np.where(y == 1)[0], n_att, replace=False)
    idx = np.concatenate([idx_ben, idx_att])
    return X.iloc[idx].reset_index(drop=True), y[idx]


In [ ]:
# ========================================
# CELULA 4: Pre-processamento + amostragem + SelectPercentile
# ========================================
NB_DATASET = "ton_iot"           # usado pela celula OOF-10 e nomes de CSV
N_TOTAL = 40000
FRAC_BENIGNO = 0.82
PERCENTIL = 60

# TON_IoT (rede): label = 1 (ataque) / 0 (benigno); type e a categoria
# multiclasse (descartada: estudo binario).
X, y = limpar_tabular(df, col_label='label', mapear={'0': 0, '1': 1, '0.0': 0, '1.0': 1})

print(f"Registros unicos limpos: {len(X)} | Features numericas: {X.shape[1]}")
print(f"Distribuicao real: benigno={sum(y==0)} ({(sum(y==0)/len(y)):.1%}) | ataque={sum(y==1)} ({(sum(y==1)/len(y)):.1%})")

X_s, y_s = amostra_estratificada(X, y, N_TOTAL, FRAC_BENIGNO)
print(f"\nAmostra final: {len(X_s)} | benigno={sum(y_s==0)} ({sum(y_s==0)/len(y_s):.1%}) | ataque={sum(y_s==1)} ({sum(y_s==1)/len(y_s):.1%})")

feature_names = list(X_s.columns)

import gc
del df, X, y
gc.collect()

X_train, X_test, y_train, y_test = train_test_split(
    X_s.values, y_s, test_size=0.3, random_state=42, stratify=y_s)
print(f"Treino: {X_train.shape[0]} | Teste: {X_test.shape[0]}")
print(f"  Teste -> Benigno: {sum(y_test==0)} | Ataque: {sum(y_test==1)}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

selector = SelectPercentile(score_func=f_classif, percentile=PERCENTIL)
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_test_sel  = selector.transform(X_test_scaled)

del X_train_scaled, X_test_scaled, X_train, X_test
gc.collect()

scores = pd.DataFrame({
    "Feature": feature_names,
    "F-Score": selector.scores_,
    "P-Value": selector.pvalues_,
    "Selecionada": selector.get_support()
}).sort_values("F-Score", ascending=False)

print("=" * 70)
print(f"  SELECTPERCENTILE ({PERCENTIL}%) - TOP 15 FEATURES")
print("=" * 70)
print(scores.head(15).to_string(index=False))

n_sel = int(sum(selector.get_support()))
print(f"\n{n_sel} features selecionadas de {len(feature_names)}:")
selecionadas = [f for f, s in zip(feature_names, selector.get_support()) if s]
print(" -> " + ", ".join(selecionadas))

plt.figure(figsize=(11, 5))
colores = ["#7b1fa2" if s else "#bdbdbd" for s in selector.get_support()]
plt.barh(scores["Feature"][:20], scores["F-Score"][:20], color=colores[:20])
plt.xlabel("F-Score (ANOVA)")
plt.title("Importancia das features (roxo = selecionadas)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# ========================================
# Helper: visualizacao PCA (2D) de balanceamento
# ========================================
from sklearn.decomposition import PCA

def plot_balance(X_res, y_res, titulo, pca=None):
    if pca is None:
        pca = PCA(n_components=2, random_state=42)
        pca.fit(X_train_sel)
    X_r = pca.transform(X_res)
    plt.figure(figsize=(6, 4))
    plt.scatter(X_r[y_res == 0, 0], X_r[y_res == 0, 1], c="steelblue", alpha=0.4, s=12, label="Benigno")
    plt.scatter(X_r[y_res == 1, 0], X_r[y_res == 1, 1], c="red", alpha=0.4, s=12, label="Ataque")
    plt.title(titulo); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ========================================
# CELULA 5: Balanceamento HIBRIDO: SMOTETomek
# ========================================
from imblearn.combine import SMOTETomek

print("Antes do balanceamento:")
print(f"  Benigno: {int(sum(y_train==0))}  Ataque: {int(sum(y_train==1))}")

smotetomek = SMOTETomek(random_state=42)
X_smotetomek, y_smotetomek = smotetomek.fit_resample(X_train_sel, y_train)
print(f"SMOTETomek: {len(X_smotetomek)} amostras "
      f"(benigno={sum(y_smotetomek==0)}, ataque={sum(y_smotetomek==1)})")

plot_balance(X_train_sel, y_train, "Original (PCA)")
plot_balance(X_smotetomek, y_smotetomek, "SMOTETomek (PCA)")


In [ ]:
# ========================================
# CELULA 6: GAN+MLP CLASSICA (referencia)
# ========================================
X_min = X_train_sel[y_train == 1]
n_min = X_min.shape[0]
n_benigno_train = int(sum(y_train == 0))
n_dim = X_train_sel.shape[1]
noise_dim = 32

print(f"Amostras de ataque: {n_min} | Dimensao: {n_dim}")

def criar_gerador():
    model = models.Sequential(name="generator_mlp")
    model.add(layers.Dense(64, input_dim=noise_dim, activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(256, activation="relu"))
    model.add(layers.Dense(n_dim, activation="tanh"))
    return model

def criar_discriminador():
    model = models.Sequential(name="discriminator_mlp")
    model.add(layers.Dense(256, input_dim=n_dim, activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation="sigmoid"))
    return model

def criar_gan():
    gerador = criar_gerador()
    discriminador = criar_discriminador()
    discriminador.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002),
                          loss="binary_crossentropy")
    discriminador.trainable = False
    gan = models.Sequential([gerador, discriminador])
    gan.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002),
                loss="binary_crossentropy")
    return gan, gerador, discriminador

gan, gerador, discriminador = criar_gan()
EPOCHS = 300
BATCH = 64
meio_batch = BATCH // 2
hist_d_loss, hist_g_loss = [], []

X_min_tf = tf.convert_to_tensor(X_min, dtype=tf.float32)
dataset = tf.data.Dataset.from_tensor_slices(X_min_tf).shuffle(1000).batch(meio_batch)

import time as _t
if "tempos_treino_generadores" not in globals():
    tempos_treino_generadores = {}
_tt0 = _t.time()
print(f"Treinando GAN+MLP por {EPOCHS} epocas...")
for epoch in range(EPOCHS):
    d_epoch = []
    g_epoch = []
    for batch in dataset:
        labels_reais = tf.ones((batch.shape[0], 1))
        ruido = tf.random.normal((batch.shape[0], noise_dim))
        falsas = gerador(ruido, training=True)
        labels_falsas = tf.zeros((batch.shape[0], 1))
        # fase discriminador: libera os pesos
        discriminador.trainable = True
        d_loss_real = discriminador.train_on_batch(batch, labels_reais)
        d_loss_fake = discriminador.train_on_batch(falsas, labels_falsas)
        d_loss = 0.5 * (d_loss_real + d_loss_fake)
        # fase gerador: congela o discriminador
        discriminador.trainable = False
        ruido = tf.random.normal((BATCH, noise_dim))
        labels_engano = tf.ones((BATCH, 1))
        g_loss = gan.train_on_batch(ruido, labels_engano)
        d_epoch.append(d_loss)
        g_epoch.append(g_loss)
    hist_d_loss.append(float(np.mean(d_epoch)))
    hist_g_loss.append(float(np.mean(g_epoch)))
    if (epoch + 1) % 50 == 0:
        print(f"  Epoca {epoch+1:3d}/{EPOCHS} | D_loss: {hist_d_loss[-1]:.4f} | G_loss: {hist_g_loss[-1]:.4f}")

tempos_treino_generadores["GAN"] = _t.time() - _tt0
print(f"Tempo de treino (GAN): {tempos_treino_generadores['GAN']:.1f} s")
n_sinteticas = n_benigno_train - n_min
ruido = tf.random.normal((n_sinteticas, noise_dim))
X_gan_sinteticas = gerador(ruido, training=False).numpy()
X_gan = np.vstack([X_train_sel, X_gan_sinteticas])
y_gan = np.hstack([y_train, np.ones(n_sinteticas)])

print(f"\nSinteticas geradas: {n_sinteticas} (para balancear 1:1)")
print(f"Dataset balanceado via GAN+MLP: {len(X_gan)} amostras "
      f"(benigno={sum(y_gan==0)}, ataque={sum(y_gan==1)})")

pca = PCA(n_components=2, random_state=42)
pca.fit(X_train_sel)
X_r = pca.transform(X_train_sel[y_train==1])
X_f = pca.transform(X_gan_sinteticas)
plt.figure(figsize=(8, 5))
plt.scatter(X_r[:, 0], X_r[:, 1], c="red", alpha=0.5, s=15, label="Ataque real")
plt.scatter(X_f[:, 0], X_f[:, 1], c="purple", alpha=0.3, s=15, label="Ataque sintetico (GAN+MLP)")
plt.title("Comparacao: Ataque real vs Sintetico (GAN+MLP classica)")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ========================================
# CELULA 7: WGAN-GP (Wasserstein GAN com Gradient Penalty)
# ========================================
n_dim = X_train_sel.shape[1]
noise_dim = 32

def criar_gerador_wgan():
    model = models.Sequential(name="generator_wgan")
    model.add(layers.Dense(64, input_dim=noise_dim, activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(256, activation="relu"))
    model.add(layers.Dense(n_dim, activation="tanh"))
    return model

def criar_critico():
    model = models.Sequential(name="critic_wgan")
    model.add(layers.Dense(256, input_dim=n_dim, activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation="linear"))
    return model

critico = criar_critico()
gerador_wgan = criar_gerador_wgan()
c_opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5, beta_2=0.9)
g_opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5, beta_2=0.9)

GP_LAMBDA = 10.0

def grad_penalty(real, fake):
    alpha = tf.random.uniform((tf.shape(real)[0], 1), 0.0, 1.0)
    interp = alpha * real + (1.0 - alpha) * fake
    with tf.GradientTape() as gp_tape:
        gp_tape.watch(interp)
        pred = critico(interp, training=True)
    grads = gp_tape.gradient(pred, interp)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((norm - 1.0) ** 2)

EPOCHS = 300; BATCH = 64; N_CRITIC = 5

X_min_tf = tf.convert_to_tensor(X_train_sel[y_train == 1], dtype=tf.float32)
n_min = X_train_sel[y_train == 1].shape[0]

def batch_real(tam):
    idx = np.random.randint(0, n_min, tam)
    return tf.gather(X_min_tf, idx)

hist_w_dist, hist_gp, hist_g_loss = [], [], []

import time as _t
if "tempos_treino_generadores" not in globals():
    tempos_treino_generadores = {}
_tt0 = _t.time()
print(f"Treinando WGAN-GP por {EPOCHS} epocas (n_critic={N_CRITIC})...")
for epoch in range(EPOCHS):
    for _ in range(N_CRITIC):
        reais = batch_real(BATCH)
        ruido = tf.random.normal((BATCH, noise_dim))
        with tf.GradientTape() as tape:
            falsas = gerador_wgan(ruido, training=True)
            d_real = critico(reais, training=True)
            d_fake = critico(falsas, training=True)
            gp = grad_penalty(reais, falsas)
            w_loss = tf.reduce_mean(d_fake) - tf.reduce_mean(d_real) + GP_LAMBDA * gp
        grads = tape.gradient(w_loss, critico.trainable_variables)
        c_opt.apply_gradients(zip(grads, critico.trainable_variables))

    ruido = tf.random.normal((BATCH, noise_dim))
    with tf.GradientTape() as tape:
        falsas = gerador_wgan(ruido, training=True)
        g_loss = -tf.reduce_mean(critico(falsas, training=True))
    grads = tape.gradient(g_loss, gerador_wgan.trainable_variables)
    g_opt.apply_gradients(zip(grads, gerador_wgan.trainable_variables))

    w_dist = tf.reduce_mean(d_real) - tf.reduce_mean(d_fake)
    hist_w_dist.append(float(w_dist.numpy()))
    hist_gp.append(float(gp.numpy()))
    hist_g_loss.append(float(g_loss.numpy()))
    if (epoch + 1) % 50 == 0:
        print(f"  Epoca {epoch+1:3d}/{EPOCHS} | W_dist: {w_dist:.4f} | GP: {gp:.4f} | G_loss: {g_loss:.4f}")

plt.figure(figsize=(10, 4))
plt.plot(hist_w_dist, color="#1e88e5", lw=1.5, label="Wasserstein distance")
plt.axhline(0, color="#888", ls="--", lw=1)
plt.xlabel("Epoca"); plt.ylabel("W_dist")
plt.title("WGAN-GP: convergencia da Wasserstein distance (sem mode collapse)")
plt.legend(); plt.tight_layout(); plt.show()

tempos_treino_generadores["WGAN-GP"] = _t.time() - _tt0
print(f"Tempo de treino (WGAN-GP): {tempos_treino_generadores['WGAN-GP']:.1f} s")
n_sinteticas = n_benigno_train - n_min
ruido = tf.random.normal((n_sinteticas, noise_dim))
X_wgan_sinteticas = gerador_wgan(ruido, training=False).numpy()
X_wgan = np.vstack([X_train_sel, X_wgan_sinteticas])
y_wgan = np.hstack([y_train, np.ones(n_sinteticas)])

print(f"\nSinteticas geradas: {n_sinteticas} (para balancear 1:1)")
print(f"Dataset balanceado via WGAN-GP: {len(X_wgan)} amostras "
      f"(benigno={sum(y_wgan==0)}, ataque={sum(y_wgan==1)})")

pca = PCA(n_components=2, random_state=42)
pca.fit(X_train_sel)
X_r = pca.transform(X_train_sel[y_train==1])
X_f = pca.transform(X_wgan_sinteticas)
plt.figure(figsize=(8, 5))
plt.scatter(X_r[:, 0], X_r[:, 1], c="red", alpha=0.5, s=15, label="Ataque real")
plt.scatter(X_f[:, 0], X_f[:, 1], c="green", alpha=0.3, s=15, label="Ataque sintetico (WGAN-GP)")
plt.title("Comparacao: Ataque real vs Sintetico (WGAN-GP)")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ========================================
# CELULA 8: cWGAN-GP (conditional WGAN-GP - usa os rotulos)
# ========================================
n_dim = X_train_sel.shape[1]
noise_dim = 32
n_benigno_train = int(sum(y_train == 0))
n_min = int(sum(y_train == 1))

X_all_tf = tf.convert_to_tensor(X_train_sel, dtype=tf.float32)
y_all_tf = tf.convert_to_tensor(y_train.reshape(-1, 1), dtype=tf.float32)

print(f"Amostras de treino (2 classes): {len(X_train_sel)} | Dimensao: {n_dim}")

def criar_gerador_cwgan():
    ruido = layers.Input(shape=(noise_dim,))
    rotulo = layers.Input(shape=(1,))
    rotulo_cast = layers.Lambda(lambda x: tf.cast(x, tf.int32))(rotulo)
    emb = layers.Embedding(2, 16)(rotulo_cast)
    emb = layers.Flatten()(emb)
    x = layers.Concatenate()([ruido, emb])
    x = layers.Dense(64, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation="relu")(x)
    saida = layers.Dense(n_dim, activation="tanh")
    return models.Model([ruido, rotulo], saida(x), name="generator_cwgan")

def criar_critico_cwgan():
    amostra = layers.Input(shape=(n_dim,))
    rotulo = layers.Input(shape=(1,))
    rotulo_cast = layers.Lambda(lambda x: tf.cast(x, tf.int32))(rotulo)
    emb = layers.Embedding(2, 16)(rotulo_cast)
    emb = layers.Flatten()(emb)
    x = layers.Concatenate()([amostra, emb])
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    saida = layers.Dense(1, activation="linear")
    return models.Model([amostra, rotulo], saida(x), name="critic_cwgan")

gerador_cwgan = criar_gerador_cwgan()
critico_cwgan = criar_critico_cwgan()

c_opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5, beta_2=0.9)
g_opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5, beta_2=0.9)

GP_LAMBDA = 10.0

def grad_penalty(real, fake, rotulos):
    alpha = tf.random.uniform((tf.shape(real)[0], 1), 0.0, 1.0)
    interp = alpha * real + (1.0 - alpha) * fake
    with tf.GradientTape() as gp_tape:
        gp_tape.watch(interp)
        pred = critico_cwgan([interp, rotulos], training=True)
    grads = gp_tape.gradient(pred, interp)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((norm - 1.0) ** 2)

EPOCHS = 300; BATCH = 64; N_CRITIC = 5

def batch_real(tam):
    idx = np.random.randint(0, len(X_train_sel), tam)
    return tf.gather(X_all_tf, idx), tf.gather(y_all_tf, idx)

hist_w_dist, hist_gp, hist_g_loss = [], [], []

import time as _t
if "tempos_treino_generadores" not in globals():
    tempos_treino_generadores = {}
_tt0 = _t.time()
print(f"Treinando cWGAN-GP por {EPOCHS} epocas (n_critic={N_CRITIC})...")
for epoch in range(EPOCHS):
    for _ in range(N_CRITIC):
        reais, rotulos = batch_real(BATCH)
        ruido = tf.random.normal((BATCH, noise_dim))
        with tf.GradientTape() as tape:
            falsas = gerador_cwgan([ruido, rotulos], training=True)
            d_real = critico_cwgan([reais, rotulos], training=True)
            d_fake = critico_cwgan([falsas, rotulos], training=True)
            gp = grad_penalty(reais, falsas, rotulos)
            w_loss = tf.reduce_mean(d_fake) - tf.reduce_mean(d_real) + GP_LAMBDA * gp
        grads = tape.gradient(w_loss, critico_cwgan.trainable_variables)
        c_opt.apply_gradients(zip(grads, critico_cwgan.trainable_variables))

    ruido = tf.random.normal((BATCH, noise_dim))
    rotulos_g = tf.random.uniform((BATCH, 1), 0, 2, dtype=tf.float32)
    with tf.GradientTape() as tape:
        falsas = gerador_cwgan([ruido, rotulos_g], training=True)
        g_loss = -tf.reduce_mean(critico_cwgan([falsas, rotulos_g], training=True))
    grads = tape.gradient(g_loss, gerador_cwgan.trainable_variables)
    g_opt.apply_gradients(zip(grads, gerador_cwgan.trainable_variables))

    w_dist = tf.reduce_mean(d_real) - tf.reduce_mean(d_fake)
    hist_w_dist.append(float(w_dist.numpy()))
    hist_gp.append(float(gp.numpy()))
    hist_g_loss.append(float(g_loss.numpy()))
    if (epoch + 1) % 50 == 0:
        print(f"  Epoca {epoch+1:3d}/{EPOCHS} | W_dist: {w_dist:.4f} | GP: {gp:.4f} | G_loss: {g_loss:.4f}")

plt.figure(figsize=(10, 4))
plt.plot(hist_w_dist, color="#1e88e5", lw=1.5, label="Wasserstein distance (condicional)")
plt.axhline(0, color="#888", ls="--", lw=1)
plt.xlabel("Epoca"); plt.ylabel("W_dist")
plt.title("cWGAN-GP: convergencia da Wasserstein distance (condicional)")
plt.legend(); plt.tight_layout(); plt.show()

tempos_treino_generadores["cWGAN-GP"] = _t.time() - _tt0
print(f"Tempo de treino (cWGAN-GP): {tempos_treino_generadores['cWGAN-GP']:.1f} s")
n_sinteticas = n_benigno_train - n_min
ruido = tf.random.normal((n_sinteticas, noise_dim))
rotulo_ataque = tf.ones((n_sinteticas, 1), dtype=tf.float32)
X_cwgan_sinteticas = gerador_cwgan([ruido, rotulo_ataque], training=False).numpy()
X_cwgan = np.vstack([X_train_sel, X_cwgan_sinteticas])
y_cwgan = np.hstack([y_train, np.ones(n_sinteticas)])

print(f"\nSinteticas geradas: {n_sinteticas} (condicionadas em classe=1 - ataque)")
print(f"Dataset balanceado via cWGAN-GP: {len(X_cwgan)} amostras "
      f"(benigno={sum(y_cwgan==0)}, ataque={sum(y_cwgan==1)})")

pca = PCA(n_components=2, random_state=42)
pca.fit(X_train_sel)
X_r = pca.transform(X_train_sel[y_train==1])
X_f = pca.transform(X_cwgan_sinteticas)
plt.figure(figsize=(8, 5))
plt.scatter(X_r[:, 0], X_r[:, 1], c="red", alpha=0.5, s=15, label="Ataque real")
plt.scatter(X_f[:, 0], X_f[:, 1], c="#6a1b9a", alpha=0.3, s=15, label="Ataque sintetico (cWGAN-GP)")
plt.title("Comparacao: Ataque real vs Sintetico (cWGAN-GP condicional)")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ========================================
# CELULA 9: CTGAN (Conditional Tabular GAN - SDV/Data to AI)
# ========================================
from ctgan import CTGAN
import time as _t
if "tempos_treino_generadores" not in globals():
    tempos_treino_generadores = {}


X_min = X_train_sel[y_train == 1]
df_min = pd.DataFrame(X_min, columns=[f"f{i}" for i in range(X_min.shape[1])])
print(f"Amostras de ataque: {len(df_min)} | Dimensao: {X_min.shape[1]}")

ctgan = CTGAN(epochs=300, batch_size=200, pac=10, verbose=False)
_tt0 = _t.time()
ctgan.fit(df_min)
tempos_treino_generadores["CTGAN"] = _t.time() - _tt0
print(f"Tempo de treino (CTGAN): {tempos_treino_generadores['CTGAN']:.1f} s")

n_sinteticas = n_benigno_train - len(df_min)
sinteticas = ctgan.sample(n_sinteticas)
X_ctgan_sinteticas = sinteticas.values
X_ctgan = np.vstack([X_train_sel, X_ctgan_sinteticas])
y_ctgan = np.hstack([y_train, np.ones(n_sinteticas)])

print(f"\nSinteticas geradas: {n_sinteticas} (para balancear 1:1)")
print(f"Dataset balanceado via CTGAN: {len(X_ctgan)} amostras "
      f"(benigno={sum(y_ctgan==0)}, ataque={sum(y_ctgan==1)})")

pca = PCA(n_components=2, random_state=42)
pca.fit(X_train_sel)
X_r = pca.transform(X_train_sel[y_train==1])
X_f = pca.transform(X_ctgan_sinteticas)
plt.figure(figsize=(8, 5))
plt.scatter(X_r[:, 0], X_r[:, 1], c="red", alpha=0.5, s=15, label="Ataque real")
plt.scatter(X_f[:, 0], X_f[:, 1], c="orange", alpha=0.3, s=15, label="Ataque sintetico (CTGAN)")
plt.title("Comparacao: Ataque real vs Sintetico (CTGAN - GMM multimodal)")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ========================================
# CELULA 10: 4 CLASSIFICADORES nos 6 cenarios
# ========================================
from tensorflow.keras.callbacks import EarlyStopping

def criar_mlp():
    model = models.Sequential(name="mlp_classifier")
    model.add(layers.Dense(128, input_dim=X_train_sel.shape[1], activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(64, activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

def criar_lstm(n_feat):
    model = models.Sequential(name="lstm_classifier")
    model.add(layers.Input(shape=(n_feat, 1)))
    model.add(layers.LSTM(64, return_sequences=True))
    model.add(layers.Dropout(0.3))
    model.add(layers.LSTM(32))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

cenarios = {
    "Original (sem balancear)": (X_train_sel, y_train),
    "SMOTETomek": (X_smotetomek, y_smotetomek),
    "GAN+MLP":    (X_gan, y_gan),
    "WGAN-GP":    (X_wgan, y_wgan),
    "cWGAN-GP":   (X_cwgan, y_cwgan),
    "CTGAN":      (X_ctgan, y_ctgan),
}

es = EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)

resultados = []

def avalia(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return acc, rec, f1, int(tn), int(fp), int(fn), int(tp)

total = len(cenarios) * 4
print(f"Treinando 4 modelos x {len(cenarios)} cenarios = {total} fits...")

for cenario, (X_tr, y_tr) in cenarios.items():
    # MLP
    modelo = criar_mlp()
    modelo.fit(X_tr, y_tr, validation_split=0.15, epochs=60, batch_size=64,
               callbacks=[es], verbose=0)
    y_pred = (modelo.predict(X_test_sel) > 0.5).astype(int).ravel()
    acc, rec, f1, tn, fp, fn, tp = avalia(y_test, y_pred)
    resultados.append({"Cenario": cenario, "Modelo": "MLP", "ACC": round(acc, 4),
                       "Recall": round(rec, 4), "F1-Score": round(f1, 4),
                       "TN": tn, "FP": fp, "FN": fn, "TP": tp})

    # XGBoost
    xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                        subsample=0.8, colsample_bytree=0.8,
                        tree_method="hist", n_jobs=-1, random_state=42,
                        eval_metric="logloss")
    xgb.fit(X_tr, y_tr)
    y_pred = xgb.predict(X_test_sel)
    acc, rec, f1, tn, fp, fn, tp = avalia(y_test, y_pred)
    resultados.append({"Cenario": cenario, "Modelo": "XGBoost", "ACC": round(acc, 4),
                       "Recall": round(rec, 4), "F1-Score": round(f1, 4),
                       "TN": tn, "FP": fp, "FN": fn, "TP": tp})

    # RandomForest
    rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_test_sel)
    acc, rec, f1, tn, fp, fn, tp = avalia(y_test, y_pred)
    resultados.append({"Cenario": cenario, "Modelo": "RandomForest", "ACC": round(acc, 4),
                       "Recall": round(rec, 4), "F1-Score": round(f1, 4),
                       "TN": tn, "FP": fp, "FN": fn, "TP": tp})

    # LSTM
    n_feat = X_train_sel.shape[1]
    lstm = criar_lstm(n_feat)
    lstm.fit(X_tr.reshape(-1, n_feat, 1), y_tr, validation_split=0.15,
             epochs=60, batch_size=64, callbacks=[es], verbose=0)
    y_pred = (lstm.predict(X_test_sel.reshape(-1, n_feat, 1)) > 0.5).astype(int).ravel()
    acc, rec, f1, tn, fp, fn, tp = avalia(y_test, y_pred)
    resultados.append({"Cenario": cenario, "Modelo": "LSTM", "ACC": round(acc, 4),
                       "Recall": round(rec, 4), "F1-Score": round(f1, 4),
                       "TN": tn, "FP": fp, "FN": fn, "TP": tp})

    print(f"  {cenario:24s} concluido")

df_res = pd.DataFrame(resultados)
df_res.insert(0, "Dataset", "TON_IoT")
print("\nDone! Linhas:", len(df_res))


In [ ]:
# ========================================
# CELULA 11: Resultados detalhados + salvamento CSV
# ========================================
print("=" * 90)
print("  RESULTADOS - TON_IoT (4 modelos x 6 cenarios)")
print("=" * 90)

pd.set_option("display.width", 200)
print(df_res.to_string(index=False))

df_res.to_csv("resultados_ton_iot.csv", index=False)
tempos_df = pd.DataFrame([{"Estrategia": k, "Tempo_treino_s": v} for k, v in tempos_treino_generadores.items()])
tempos_df.to_csv("tempos_treino_ton_iot.csv", index=False)
try:
    from google.colab import files
    files.download("tempos_treino_ton_iot.csv")
except Exception:
    pass
print("=== TEMPOS DE TREINO (s) ===\n" + tempos_df.to_string(index=False))
try:
    from google.colab import files
    files.download("resultados_ton_iot.csv")
except Exception:
    print("\n(Se este ambiente nao for Colab, baixe manualmente resultados_ton_iot.csv)")

print("\nANALISE DE FALSOS NEGATIVOS (FN - ataque que escapa) e FALSOS POSITIVOS (FP):")
for modelo in ["MLP", "XGBoost", "RandomForest", "LSTM"]:
    sub = df_res[df_res["Modelo"] == modelo]
    print(f"\n  == Modelo: {modelo} ==")
    for _, row in sub.iterrows():
        print(f"    {row['Cenario']:24s} | ACC={row['ACC']:.4f} | F1={row['F1-Score']:.4f} "
              f"| TP={row['TP']:5d} FP={row['FP']:5d} FN={row['FN']:4d} TN={row['TN']:5d}")

print("\n" + "=" * 90)
print("  RANKING GERAL (por F1, com FN como desempate)")
print("=" * 90)
rank = df_res.sort_values(["F1-Score", "FN"], ascending=[False, True])
print(rank.to_string(index=False))

melhor = rank.iloc[0]
print("\nMELHOR CONFIGURACAO GERAL:")
print(f"  Dataset:  {melhor['Dataset']}")
print(f"  Modelo:   {melhor['Modelo']}")
print(f"  Cenario:  {melhor['Cenario']}")
print(f"  F1={melhor['F1-Score']} | Recall={melhor['Recall']} | ACC={melhor['ACC']}")
print(f"  TP={melhor['TP']} FP={melhor['FP']} FN={melhor['FN']} TN={melhor['TN']}")


In [ ]:
# ========================================
# CELULA 12: Figuras comparativas
# ========================================
cen_ord = ["Original (sem balancear)", "SMOTETomek", "GAN+MLP", "WGAN-GP", "cWGAN-GP", "CTGAN"]
cen_simples = ["Original", "SMOTETomek", "GAN+MLP", "WGAN-GP", "cWGAN-GP", "CTGAN"]
cores_c = dict(zip(cen_simples, ["#bdbdbd", "#7b1fa2", "#8e24aa", "#6a1b9a", "#4a148c", "#2e7d32"]))
cores_c["Original (sem balancear)"] = "#bdbdbd"
modelos = ["MLP", "XGBoost", "RandomForest", "LSTM"]

fig, axes = plt.subplots(2, 2, figsize=(17, 10))
for i, mdl in enumerate(modelos):
    ax = axes[i // 2][i % 2]
    sub = df_res[df_res["Modelo"] == mdl]
    pal = [cores_c.get(c, "#bdbdbd") for c in sub["Cenario"]]
    sns.barplot(data=sub, x="Cenario", y="F1-Score", palette=pal, ax=ax)
    ax.set_title(f"F1-Score - {mdl} (TON_IoT)")
    ax.set_ylim(sub["F1-Score"].min() - 0.02, 1.0)
    ax.tick_params(axis="x", rotation=25)
    for j, v in enumerate(sub["F1-Score"]):
        ax.text(j, v + 0.002, f"{v:.3f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(17, 10))
for i, mdl in enumerate(modelos):
    ax = axes[i // 2][i % 2]
    sub = df_res[df_res["Modelo"] == mdl]
    pal = ["#b71c1c" if c != "CTGAN" else "#2e7d32" for c in sub["Cenario"]]
    sns.barplot(data=sub, x="Cenario", y="FN", palette=pal, ax=ax)
    ax.set_title(f"FN (ataque que escapou) - {mdl} (TON_IoT)")
    ax.tick_params(axis="x", rotation=25)
    for j, v in enumerate(sub["FN"]):
        ax.text(j, v + max(1, int(0.01 * v)), f"{v}", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()

melhor_modelo = df_res.sort_values(["F1-Score", "FN"], ascending=[False, True]).iloc[0]["Modelo"]
sub = df_res[df_res["Modelo"] == melhor_modelo]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for i, cenario in enumerate(cenarios):
    row = sub[sub["Cenario"] == cenario].iloc[0]
    cm = np.array([[row["TN"], row["FP"]], [row["FN"], row["TP"]]])
    cmap = "Greens" if cenario == "CTGAN" else "Purples"
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=axes[i // 3][i % 3],
                xticklabels=["Benigno", "Ataque"], yticklabels=["Benigno", "Ataque"], cbar=False)
    axes[i // 3][i % 3].set_title(f"{cenario}\nACC={row['ACC']:.4f} | F1={row['F1-Score']:.4f}")
    axes[i // 3][i % 3].set_xlabel("Predito"); axes[i // 3][i % 3].set_ylabel("Real")
plt.suptitle(f"Matrizes de Confusao - {melhor_modelo} (TON_IoT)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nREDUCAO DE FN vs ORIGINAL (percentual):")
for modelo in modelos:
    sub = df_res[df_res["Modelo"] == modelo]
    base = sub[sub["Cenario"] == "Original (sem balancear)"].iloc[0]
    best = sub.sort_values(["F1-Score", "FN"], ascending=[False, True]).iloc[0]
    red_fn = 100 * (1 - best["FN"] / base["FN"]) if base["FN"] > 0 else float("nan")
    red_fp = 100 * (1 - best["FP"] / base["FP"]) if base["FP"] > 0 else float("nan")
    print(f"  {modelo:12s} | {best['Cenario']:20s} | FN {base['FN']} -> {best['FN']} "
          f"(-{red_fn:.1f}%) | FP {base['FP']} -> {best['FP']} (-{red_fp:.1f}%)")


## 13. Discussao

**ANALISE CRITICA DOS RESULTADOS**

1. **O objetivo e o mesmo das aulas anteriores: reduzir o Falso Negativo** — o ataque que escapa da deteccao. Para cada um dos 4 classificadores (MLP, XGBoost, RandomForest, LSTM), avalie qual cenario de balanceamento atingiu o **menor FN** mantendo o maior F1.

2. **Interpretacao por modelo:** o MLP (Keras) replica exatamente o pipeline do IoT-23. XGBoost e RandomForest sao os baselines classicos *single-model* que dominam a avaliacao de NIDS — tendem a ja ter FN baixo mesmo sem balanceamento (mais robustos a dados desbalanceados). O LSTM (fluxo de features como serie temporal) e o representante *deep temporal*. Compare se a ordem dos cenarios muda entre modelos — se o CTGAN ganha em todos, o efeito e **atribuivel ao balanceamento**, nao ao classificador.

3. **Transparencia metodologica:** o TON_IoT aqui usado e o dataset real oficial (fonte publica). A amostragem estratificada para 82/18 e um protocolo declarado para comparar com o IoT-23 — observe de que lado a classe ataque esta no conjunto original e discuta o efeito esperado.

4. **Interpretacao dos falsos positivos:** alem do FN, observe o FP (alarme falso que gera fadiga de alerta no SOC). O balanceamento generativo reduziu FN e FP em **todos** os datasets? Isso fortalece o argumento do artigo.

**COMO USAR NO ARTIGO:** o CSV `resultados_ton_iot.csv` baixado acima e consumido pela **aula de consolidacao** (`aula_iot23_consolidacao.ipynb`), que junta os 3 datasets e gera as tabelas e figuras finais.


## 14. Conclusao e como reproduzir

- Nesta aula voce rodou a **mesma ideia das aulas 1–5** (6 cenarios de balanceamento: original, SMOTETomek, GAN+MLP, WGAN-GP, cWGAN-GP, CTGAN) sobre um **dataset real de NIDS** (TON_IoT), agora com **4 classificadores** (MLP, XGBoost, RandomForest, LSTM) e sempre reportando **FN/FP/TP/TN** sobre o mesmo teste estratificado.

- **Para reproduzir:**
  1. Rode este notebook do inicio ao fim (Runtime -> Run all). A execucao leva ~30–60 min no Colab (GPU recomendada).
  2. Ao final, o CSV `resultados_ton_iot.csv` e baixado automaticamente (ou via painel de arquivos).
  3. Guarde o CSV na mesma pasta dos demais e rode em seguida a **aula de consolidacao** para montar a comparacao final do artigo.


## [OOF-10] Robustez — StratifiedKFold(10) out-of-fold (cadeia-chave LSTM)
Protocolo honesto, sem vazamento:
- StratifiedKFold(10, shuffle, seed 42) sobre a amostra estratificada real (X_s/y_s);
- em cada fold: scaler+seletor+balanceador ajustados SOMENTE no treino do fold; GANs (gerador_wgan/gerador_cwgan) usadas em INFERÊNCIA (sem re-treino por fold — custo proibido, documentado como limitação honesta);
- LSTM (criar_lstm) treinada no fold; predição OOF no valid do fold;
- agrega predições OOF dos 10 folds → média±desvio ACC/Recall/F1, soma FN/FP, tempo por fold.
Saída (download no Colab): resultados_oof10_ton_iot.csv + tempos_oof10_ton_iot.csv


In [ ]:
# ============================================================
# [OOF-10] ROBUSTEZ — StratifiedKFold(10) out-of-fold, cadeia-chave LSTM
# ============================================================
# Protocolo honesto (nucleo da tese, sem vazamento):
#   * StratifiedKFold(10, shuffle, random_state=42) sobre a AMOSTRA
#     estratificada real (X_s, y_s);
#   * a CADA fold: scaler (StandardScaler) + seletor (SelectPercentile
#     f_classif) + balanceador ajustados SOMENTE no treino do fold;
#     o VALID do fold passa por scaler/seletor ajustados no treino
#     (nunca no valido) -> sem vazamento;
#   * cenarios de balanceamento, CASO o respectivo gerador ja tenha sido
#     TREINADO nas celulas GAN acima (inferencia, training=False):
#       Original  | SMOTETomek | WGAN-GP | cWGAN-GP
#     GANs NAO sao re-treinadas por fold (documentado: custo proibido;
#     protocolo alternativo qcd na secao Robustez do artigo);
#   * LSTM (criar_lstm) treinada no fold; predicao OUT-OF-FOLD no valid;
#   * agrega predicoes dos 10 folds -> media += DP de ACC/Recall/F1 e
#     soma de FN/FP/TN/TP por cenario.
# Saidas (CSVs, baixaveis no Colab):
#   resultados_oof10_<NB_DATASET>.csv   (por fold + agregado)
#   tempos_oof10_<NB_DATASET>.csv
# ============================================================
import time as _t_oof10
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                             confusion_matrix)

OOF10_FOLDS   = 10
OOF10_SEED    = 42
OOF10_EPOCHS  = int(globals().get("EPOCHS", 100))
OOF10_PAT     = 6
OOF10_NFOLDS  = OOF10_FOLDS

def _oof_gerador(nome_base):
    """Retorna o gerador real ja treinado (WGAN-GP ou cWGAN-GP), se
    existir em globals(); senao None (fallback honesto: sem balancear)."""
    for k in (nome_base, nome_base + "_gp", nome_base + "_final",
              "gerador_" + nome_base, nome_base + "_treinado"):
        if k in globals():
            return globals()[k]
    return None

def _oof_sinteticas(X_fake_tr, y_fake_tr, cen, n_sint, noise_d):
    """Gera n_sint sinteticas do minoritario REAL do fold-treino com o
    gerador treinado acima (inferencia). None => indisponivel."""
    if cen == "WGAN-GP":
        g = _oof_gerador("gerador_wgan")
        if g is None:
            return None
        ruido = tf.random.normal((n_sint, noise_d))
        return g(ruido, training=False).numpy()
    if cen == "cWGAN-GP":
        g = _oof_gerador("gerador_cwgan")
        if g is None:
            return None
        ruido = tf.random.normal((n_sint, noise_d))
        rotulo = tf.ones((n_sint, 1), dtype=tf.float32)
        return g([ruido, rotulo], training=False).numpy()
    return None

CENARIOS_OOF = ["Original (sem balancear)", "SMOTETomek",
                "WGAN-GP", "cWGAN-GP"]

X_oof = np.asarray(X_s, dtype=np.float32)
y_oof = y_s
print(f"[OOF-10] dataset={NB_DATASET!r} | amostra real n={len(X_oof)} | "
      f"ataque={int(y_oof.sum())}")

skf_oof = StratifiedKFold(n_splits=OOF10_FOLDS, shuffle=True,
                          random_state=OOF10_SEED)
linhas_oof = []
tempos_oof = []
_t0_oof = _t_oof10.time()

# deteccao do noise_dim real do notebook (fallback 32)
noise_d = 32
for k in ("noise_dim", "ruido_dim", "NOISE_DIM", "noise_d"):
    if k in globals():
        noise_d = int(globals()[k])
        break

for fold, (idx_tr, idx_va) in enumerate(
        skf_oof.split(X_oof, y_oof), start=1):
    _t_fold = _t_oof10.time()
    X_tr_f = X_oof[idx_tr]; y_tr_f = y_oof[idx_tr]
    X_va_f = X_oof[idx_va]; y_va_f = y_oof[idx_va]

    sc_oof = StandardScaler().fit(X_tr_f)
    X_tr_s = sc_oof.transform(X_tr_f)
    X_va_s = sc_oof.transform(X_va_f)
    sel_oof = SelectPercentile(f_classif, percentile=60).fit(X_tr_s, y_tr_f)
    X_tr_sel = sel_oof.transform(X_tr_s)
    X_va_sel = sel_oof.transform(X_va_s)
    n_feat = X_tr_sel.shape[1]

    for cen in CENARIOS_OOF:
        X_bal = X_tr_sel; y_bal = y_tr_f
        if cen == "SMOTETomek":
            smt_oof = SMOTETomek(random_state=OOF10_SEED)
            X_bal, y_bal = smt_oof.fit_resample(X_tr_sel, y_tr_f)
        elif cen in ("WGAN-GP", "cWGAN-GP"):
            n_ata = int(y_tr_f.sum()); n_ben = int((1 - y_tr_f).sum())
            n_sint = n_ben - n_ata
            if n_sint > 0:
                X_fake = _oof_sinteticas(X_tr_sel, y_tr_f, cen, n_sint,
                                         noise_d)
                if X_fake is not None:
                    X_fake = np.asarray(X_fake, dtype=np.float32)
                    if X_fake.ndim == 3:
                        X_fake = X_fake.reshape(X_fake.shape[0], -1)
                    X_bal = np.vstack([X_tr_sel, X_fake[:, :n_feat]])
                    y_bal = np.hstack([y_tr_f, np.ones(n_sint)])
        a = criar_lstm(n_feat)
        es_oof = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=OOF10_PAT,
            restore_best_weights=True)
        a.fit(X_bal.reshape(-1, n_feat, 1), y_bal, validation_split=0.15,
              epochs=OOF10_EPOCHS, batch_size=64, callbacks=[es_oof],
              verbose=0)
        yp = (a.predict(X_va_sel.reshape(-1, n_feat, 1), verbose=0)
              > 0.5).astype(int).ravel()
        tn, fp, fn, tp = confusion_matrix(y_va_f, yp).ravel()
        linhas_oof.append({"Fold": fold, "Cenario": cen,
                           "ACC": round(accuracy_score(y_va_f, yp), 4),
                           "Recall": round(recall_score(y_va_f, yp), 4),
                           "F1-Score": round(f1_score(y_va_f, yp), 4),
                           "TN": int(tn), "FP": int(fp),
                           "FN": int(fn), "TP": int(tp)})
        print(f"  fold {fold:2d} | {cen:26s} | "
              f"F1={f1_score(y_va_f, yp):.4f} | FN={int(fn)}")
    tempos_oof.append({"Fold": fold, "Tempo_s": round(_t_oof10.time() - _t_fold, 1)})

pd.DataFrame(linhas_oof).to_csv(f"resultados_oof10_{NB_DATASET}.csv",
                                index=False)
pd.DataFrame(tempos_oof).to_csv(f"tempos_oof10_{NB_DATASET}.csv",
                                index=False)

print("\n" + "=" * 78)
print("  OOF-10 AGREGADO (media +/- DP) por cenario")
print("=" * 78)
df = pd.DataFrame(linhas_oof)
agg = (df.groupby("Cenario")[["ACC", "Recall", "F1-Score"]]
       .agg(["mean", "std"]).round(4))
agg.columns = ["ACC_m", "ACC_dp", "Recall_m", "Recall_dp",
               "F1_m", "F1_dp"]
fnsum = df.groupby("Cenario")["FN"].sum().rename("FN_total")
print(pd.concat([agg, fnsum], axis=1).to_string())
print(f"\ntempo total: {_t_oof10.time() - _t0_oof:.1f} s")
print(f"CSV salvo: resultados_oof10_{NB_DATASET}.csv (baixe no Colab)")
try:
    from google.colab import files
    files.download(f"resultados_oof10_{NB_DATASET}.csv")
    files.download(f"tempos_oof10_{NB_DATASET}.csv")
except Exception:
    pass
